<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# K-Means Clustering with Seeds Data - Solution

---

In [ ]:
%matplotlib inline 

import pandas as pd
import numpy as np
from sklearn import cluster
from sklearn import metrics
from sklearn.metrics import pairwise_distances
import matplotlib.pyplot as plt
import matplotlib
matplotlib.style.use('ggplot') 

import seaborn as sns

### 1. Import the data

In [ ]:
seeds = pd.read_csv("../data/seeds.csv")

In [ ]:
# Taking a peek
seeds.head()

### 2. Do some EDA of relationships between features.

In [ ]:
# Plot the Data to see the distributions/relationships
import seaborn as sns

# Plot without the "species" hue.
sns.pairplot(seeds)

Can our naked eye see any clusters within these scatter plots?

- *groove_lenght vs. compactness looks like 2 clusters*  
- *perimiter vs. groove_length maybe 3 clusters*

In [ ]:
# Check for nulls
seeds.isnull().sum()
# there is a value for every position in the DF

In [ ]:
# Look at the real species labels.
sns.pairplot(data=seeds, hue='species')
# classes appear to have a similar number of samples.
# Blue consistently looks like the divisor between the green and red classes.

Remember, clustering is a unsupervised learning method so known classes will never be a thing.  In this situation we can see that the `perimiter` vs. `groove_length` is a good visualization to view the proper classes class, and we can use later to compare the results of clustering to a true value.

In [ ]:
seeds.species.value_counts()
# all classes are equally distributed. 

In [ ]:
# Check datatypes
seeds.dtypes
# We got an odd-ball, that species guy.

### 3. Prepare the data for clustering

1. Remove the `species` column. We will see if the clusters from K-Means end up like the actual species.
2. Put the features on the same scale.

In [ ]:
# drop 'species', which is currently acting as a target (categorical)
X = seeds.drop('species', axis = 1)
y = seeds.species

### 4. Clustering with K-Means

- Cluster the data to our our target groups.
- We know that there are 3 actual classes. However, in an actual situation in which we used clustering we would have no idea. Lets initally try using the default K for `KMeans`(8).

In [ ]:
from sklearn.cluster import KMeans

# 2 Clusters
k_mean = KMeans()
k_mean.fit(X)

### 5. Get the labels and centroids for out first clustering model.

In [ ]:
# Labels and centroids for 8 Clusters
labels = k_mean.labels_
print(labels)
clusters = k_mean.cluster_centers_
clusters

### 6. Compute the silouette score and visually examine the results of the 8 clusters. 

_(pairplot with hue)_

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_score(X, labels)

In [ ]:
# Considering silhouette is on a scale of -1 to 1, 0.35 isnt too bad.

In [ ]:
# visually examine the cluster that have been created
X_8 = seeds.drop('species', axis=1)
X_8['clusters']=labels

sns.pairplot(data=X_8, hue='clusters')

### 7. Repeat steps #4 and #6 with two selected or random K values and compare the results to the k=8 model.

In [ ]:
import random

random.randint(1,25), random.randint(1,25)

In [ ]:
# 4 Clusters
k_mean4 = KMeans(n_clusters=4)
k_mean4.fit(X)
labels_4 = k_mean4.labels_
silhouette_score(X, labels_4)

In [ ]:
X_4 = seeds.drop('species', axis=1)
X_4['clusters']=labels_4

sns.pairplot(data=X_4, hue='clusters')

In [ ]:
# k=4 was the best performing of the Ks i tested
# looks like scatter plot of perimeter vs. asymmetry_coeff
# distingusihed the cluster the best.

In [ ]:
# 6 Clusters
k_mean6 = KMeans(n_clusters=6)
k_mean6.fit(X)
labels_6 = k_mean6.labels_
silhouette_score(X, labels_6)

In [ ]:
X_6 = seeds.drop('species', axis=1)
X_6['clusters']=labels_6

sns.pairplot(data=X_6, hue='clusters')

In [ ]:
# perimeter vs asymmetry_coeff & area vs. asymmetry_coeff
# distiguished the clusters the best visually.

### 8. Build a function to find the optimal number of clusters using silhouette score as the criteria.
1. Function should accept a range and a dataframe as arguments
2. Returns the optimal K value, associate silhoutte and scaling method.
3. Your function should also consider the scaled results of the data. 
    - `normalize`, `StandardScaler`, `MinMaxScaler`


Once you have found the optimal K and version of the data, visualize the clusters.





In [ ]:
#necessary processing imports
from sklearn.preprocessing import normalize, StandardScaler, MinMaxScaler

In [ ]:
# create dataframe to append info too
results = pd.DataFrame(columns = ['k','silhouette','processing'])


def cluster(ran, data, version):
    for k in ran:
        k_means = KMeans(n_clusters=k)
        k_means.fit(data)
        labels = k_means.labels_
        score = silhouette_score(data, labels)
        results.loc[len(results)]=['c'+str(k), score, version]

In [ ]:
def opt_cluster(ran, data):
    cluster(ran, data, 'default')
    
    # normalized version
    Xn = normalize(data)
    cluster(ran, Xn, 'normalized')
    
    # standard scale version
    SS = StandardScaler()
    Xs = SS.fit_transform(data)
    cluster(ran, Xs, 'standard_scaler')
    
    # minmax scale version
    MM = MinMaxScaler()
    Xmm = MM.fit_transform(data)
    cluster(ran, Xmm, 'min_max_scaler')

    return results.loc[results['silhouette'].idxmax()]


In [ ]:
ran = list(range(2,12))

opt_cluster(ran,X)

In [ ]:
# build the model with the found optimal parameters
k_mean_opt = KMeans(n_clusters=2)
k_mean_opt.fit(X)
labels_opt = k_mean_opt.labels_

# no preprocessing required since default was the highest silouette
X_opt = seeds.drop('species', axis=1)

X_opt['clusters']=labels_opt
sns.pairplot(data=X_opt, hue='clusters')


As we can see the difference between our exploratory analysis with the original data and the results of finding an optimal clustering model, silouette score can be an untrustworthy means of evaluating a cluster.  As this is an unsupervised model it will just to conclusions that we as humans may know to not be true.   


 In this situation the non-processed data performed better than the processed, but there are a  variety of cases where the opposite is true.  Preprocessing and scaling is an extremely important step when clustering in order to negative the huge affects outliers could have on clusters. 
 
One of the more highly recommended scaling tactics is `MinMax`, because you can somewhat control the range / magnitude of your scale within multiple dimensions to augment your data in ways that could be more beneficial to the convergence of K-Means.